<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h1 align="right" dir="rtl">هفته 11 — تمرین کلاسی: به‌کارگیری Data Augmentation</h1>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>سؤال اصلی:</strong> با داشتن دو مسئله واقعی از client و فقط تعداد کمی image دارای label برای هر کدام، آیا می‌توانید به‌جای copy-paste کردن یک روش آماده، یک augmentation strategy را <strong>طراحی کنید، برایش دلیل بیاورید و با شواهد نشان دهید که درست کار می‌کند</strong>؟</p>
</blockquote>
<h2 align="right" dir="rtl">نمای کلی Session</h2>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" dir="rtl"><strong>مدت:</strong> حدود 3 ساعت</li>

<li align="right" dir="rtl"><strong>Framework:</strong> PyTorch و Torchvision</li>
<li align="right" dir="rtl"><strong>پیش‌نیاز:</strong> Session 1 (small-data baselines و overfitting) و Session 2 (augmentation)</li>
<li align="right" dir="rtl"><strong>Datasetهای امروز:</strong> بخشی از <strong>CIFAR-10</strong> (عکس‌های طبیعی) و بخشی از <strong>MNIST</strong> (اعداد دست‌نویس).</li>
</ul>
<hr/>
<h2 align="right" dir="rtl">هفته اول شما در PixelWorks AI</h2>
<p align="right" dir="rtl">تبریک! شما به‌تازگی به‌عنوان یک junior ML engineer شروع به کار کرده‌اید. امروز صبح دو ticket وارد صف کارتان شده است. هر دو client عجله دارند، هر دو dataset کوچک هستند و tech lead شما فقط شواهد می‌خواهد، نه نظر شخصی.</p>
<p align="right" dir="rtl"><strong>🎫 Ticket #1 — SnapSort.</strong> یک startup برای مرتب‌کردن عکس‌ها به کاربران اجازه می‌دهد با موبایل از اشیای روزمره عکس بگیرند؛ عکس‌ها ممکن است کج، crop‌شده، کم‌نور یا به‌صورت دستی گرفته شده باشند. برای هر class فقط چندصد image دارای label دارند. شنیده‌اند که «augmentation مشکل small data را حل می‌کند» و از شما می‌خواهند این کار را درست انجام دهید.</p>
<p align="right" dir="rtl"><strong>🎫 Ticket #2 — SecureDigits.</strong> یک شرکت fintech اعداد دست‌نویس را از فرم‌های کاغذی scan‌شده می‌خواند. contractor قبلی یک augmentation «استاندارد» به pipeline اضافه کرده است. بعد از deployment، accuracy روی فرم‌های واقعی بدتر شده و هیچ‌کس دلیلش را نمی‌داند. از شما خواسته شده مشکل را بررسی کنید.</p>
<p align="right" dir="rtl">تا پایان این تمرین، تمام مراحل یک augmentation workflow را انجام می‌دهید: بررسی baseline، طراحی policy همراه با دلیل، پیدا کردن policyای که بی‌سروصدا labelها را خراب می‌کند، بررسی اثر واقعی augmentation با یک controlled experiment و در نهایت نوشتن دو memo کوتاه که یک ML engineer در دنیای واقعی ممکن است مجبور باشد ارسال کند.</p>
<h3 align="right" dir="rtl">در پایان چه چیزهایی باید تحویل دهید؟</h3>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>code cellهای کامل‌شده (همه بخش‌های <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;"># TODO</code>)</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>decision tableهای تکمیل‌شده و پاسخ‌های نوشتاری (markdown cellهای دارای <strong>پاسخ شما:</strong>)</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>دو comparison plot: SnapSort baseline-vs-augmented و SecureDigits safe-vs-harmful</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>دو memo کوتاه برای client در Part 5 و Part 4</li>
</ul>
<p align="right" dir="rtl">شروع کنید.</p>

</div>


In [ ]:
# Setup — same conventions as Session 1 & 2, nothing to change here.
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

SEED = 42
DATA_ROOT = Path("data")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h1 align="right" dir="rtl">Part 0 — Warm-Up: قبل از ساختن، Diagnose کنید (≈15 دقیقه)</h1>
<p align="right" dir="rtl">قبل از اینکه سراغ داده جدید client بروید، tech lead می‌خواهد مطمئن شود می‌توانید یک training log را <strong>بدون داشتن plot</strong> بخوانید؛ یعنی scriptای که training runهای شبانه را بررسی کند و مشکل‌ها را به‌صورت خودکار flag کند. فرض کنید در این محیط plotting library در اختیار ندارید.</p>
<p align="right" dir="rtl">در ادامه چهار training history مصنوعی از چهار پروژه قبلی و فرضی می‌بینید. ساختار هرکدام دقیقاً شبیه objectهای <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">history</code> در Session 1 و Session 2 است و شامل <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_loss</code>، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">val_loss</code>، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_accuracy</code> و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">val_accuracy</code> می‌شود؛ برای هر epoch یک مقدار وجود دارد.</p>
<p align="right" dir="rtl"><strong>Q0.1 (اول پیش‌بینی کنید):</strong> قبل از اجرای هر چیزی، چهار accuracy curve زیر را سریع بررسی کنید. فکر می‌کنید function شما کدام مورد را <strong>اشتباه</strong> تشخیص می‌دهد و چرا؟ حدس خود را در یک جمله بنویسید.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


In [ ]:
# Four fictional training logs. Feel free to plot them if it helps you reason,
# but the point of this drill is to reason about numbers directly.

scenario_A_underfitting = {
    "train_accuracy": [0.35, 0.40, 0.42, 0.45, 0.46, 0.47, 0.47, 0.48, 0.48, 0.49],
    "val_accuracy":   [0.33, 0.38, 0.40, 0.43, 0.45, 0.46, 0.46, 0.47, 0.47, 0.48],
    "train_loss":     [1.80, 1.60, 1.50, 1.42, 1.38, 1.35, 1.34, 1.33, 1.32, 1.31],
    "val_loss":       [1.82, 1.63, 1.53, 1.45, 1.41, 1.38, 1.37, 1.36, 1.35, 1.34],
}

scenario_B_overfitting = {
    "train_accuracy": [0.40, 0.55, 0.68, 0.78, 0.86, 0.91, 0.94, 0.96, 0.97, 0.98],
    "val_accuracy":   [0.38, 0.52, 0.60, 0.63, 0.64, 0.63, 0.62, 0.61, 0.60, 0.59],
    "train_loss":     [1.70, 1.30, 0.95, 0.68, 0.48, 0.34, 0.24, 0.17, 0.13, 0.10],
    "val_loss":       [1.72, 1.35, 1.08, 0.95, 0.90, 0.92, 0.98, 1.05, 1.12, 1.20],
}

scenario_C_good_fit = {
    "train_accuracy": [0.45, 0.60, 0.72, 0.80, 0.85, 0.88, 0.90, 0.91, 0.92, 0.93],
    "val_accuracy":   [0.43, 0.58, 0.69, 0.76, 0.80, 0.83, 0.85, 0.86, 0.87, 0.87],
    "train_loss":     [1.60, 1.15, 0.85, 0.65, 0.52, 0.44, 0.39, 0.35, 0.32, 0.30],
    "val_loss":       [1.63, 1.20, 0.90, 0.70, 0.58, 0.50, 0.46, 0.43, 0.41, 0.40],
}

scenario_D_confident_mistakes = {
    # Accuracy looks almost identical to a "good fit" — barely any gap.
    "train_accuracy": [0.50, 0.60, 0.66, 0.69, 0.70, 0.70, 0.71, 0.70, 0.70, 0.71],
    "val_accuracy":   [0.48, 0.58, 0.64, 0.68, 0.69, 0.70, 0.70, 0.69, 0.70, 0.70],
    "train_loss":     [1.30, 1.00, 0.80, 0.65, 0.55, 0.48, 0.42, 0.38, 0.34, 0.30],
    # ...but validation LOSS keeps climbing after epoch 5.
    "val_loss":       [1.30, 1.10, 0.95, 0.85, 0.82, 0.85, 0.90, 0.98, 1.05, 1.15],
}

all_scenarios = {
    "A - underfitting?": scenario_A_underfitting,
    "B - overfitting?": scenario_B_overfitting,
    "C - good fit?": scenario_C_good_fit,
    "D - ???": scenario_D_confident_mistakes,
}

In [ ]:
def diagnose_fit(history, gap_threshold=0.10, weak_threshold=0.65):
    '''Classify a training history using ONLY the final-epoch accuracies.

    Return one of: "underfitting", "overfitting", "good fit".

    Hints:
      - final_train = history["train_accuracy"][-1]
      - final_val   = history["val_accuracy"][-1]
      - If both are below `weak_threshold` -> "underfitting"
      - Else if (final_train - final_val) > gap_threshold -> "overfitting"
      - Else -> "good fit"
    '''
    # TODO: implement diagnose_fit
    raise NotImplementedError


for name, history in all_scenarios.items():
    print(f"{name:>18s} -> {diagnose_fit(history)}")

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>Q0.2:</strong> cell بالا را اجرا کنید. با توجه به اتفاقی که در طول epochها می‌افتد، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">diagnose_fit</code> کدام scenario را به شکلی classify کرده که به‌نظر <strong>اشتباه</strong> یا ناقص می‌رسد؟ <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">diagnose_fit</code> اجازه ندارد به کدام عدد نگاه کند؟ در حد توضیح متنی و بدون نیاز به recode کردن بگویید function را چطور تغییر می‌دهید تا این حالت را تشخیص دهد.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h1 align="right" dir="rtl">Part 1 — Ticket #1: SnapSort، ساخت Baseline قبل از هر Augmentation (≈35 دقیقه)</h1>
<p align="right" dir="rtl"><strong>متن ticket از زبان client:</strong></p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl">«برای هر category حدود 60 عکس دارای label داریم؛ cat، dog، car، ship و چیزهایی از این جنس که مردم با موبایل عکس می‌گیرند. این هفته یک proof-of-concept classifier می‌خواهیم. فعلاً فقط چیزی بسازید که کار کند؛ بعداً سراغ روش‌های پیشرفته‌تر می‌رویم.»</p>
</blockquote>
<p align="right" dir="rtl">از Session 1 می‌دانید هدف یک proof-of-concept baseline این نیست که چشمگیر باشد؛ هدف این است که شواهد صادقانه‌ای از سختی واقعی مسئله به دست بدهد. از <strong>CIFAR-10</strong> استفاده می‌کنیم و عمداً بیشتر داده را کنار می‌گذاریم تا وانمود کنیم برای هر class فقط تعداد کمی image دارای label در اختیار داریم. این وضعیت به چیزی که در شروع کار با dataset یک client جدید اتفاق می‌افتد بسیار نزدیک است.</p>

</div>


In [ ]:
# Utility code (given) — subsampling a full torchvision dataset into a
# small, class-balanced pool, and a Dataset wrapper that applies a chosen
# transform on the fly. You'll reuse both utilities again in Part 3.

def make_balanced_indices(targets, class_ids, n_per_class, seed=42):
    '''Return a shuffled list of indices with exactly n_per_class examples
    per class in class_ids.'''
    rng = random.Random(seed)
    targets = list(targets)
    chosen = []
    for c in class_ids:
        idx_c = [i for i, t in enumerate(targets) if t == c]
        rng.shuffle(idx_c)
        chosen.extend(idx_c[:n_per_class])
    rng.shuffle(chosen)
    return chosen


def make_val_test_split(targets, class_ids, n_val, n_test, seed=42):
    '''Return two DISJOINT, class-balanced index lists drawn from the same
    pool — one for validation, one for test. Disjoint on purpose: a val/test
    leak would quietly invalidate every experiment you run today.'''
    rng = random.Random(seed)
    targets = list(targets)
    val_indices, test_indices = [], []
    for c in class_ids:
        idx_c = [i for i, t in enumerate(targets) if t == c]
        rng.shuffle(idx_c)
        val_indices.extend(idx_c[:n_val])
        test_indices.extend(idx_c[n_val:n_val + n_test])
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)
    return val_indices, test_indices


class TransformSubset(Dataset):
    '''Wraps a base dataset + a fixed list of indices, applying `transform`
    each time an item is fetched. This is the same trick Session 2 used when
    it built `train_baseline_dataset` and `train_augmented_dataset` from the
    SAME folder with two different transforms — here we do it with indices
    instead of two copies of a folder.

    `label_map` (given, not the focus of this exercise) remaps the dataset's
    original class ids (e.g. CIFAR-10's "cat" = 3) down to a compact
    0..num_classes-1 range, which CrossEntropyLoss requires whenever we only
    use a handful of the original classes.'''

    def __init__(self, base_dataset, indices, transform, label_map=None):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform
        self.label_map = label_map

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.base_dataset[self.indices[idx]]
        if self.label_map is not None:
            label = self.label_map[label]
        # TODO: apply self.transform to `image` and return (transformed_image, label)
        raise NotImplementedError

In [ ]:
# Download CIFAR-10 once, WITHOUT any transform yet — we want raw PIL images
# so we can attach different transforms to the same underlying pictures later.
raw_train_full = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=None)
raw_test_full = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=None)

snapsort_classes = ["cat", "dog", "automobile", "ship"]
snapsort_class_ids = [raw_train_full.classes.index(c) for c in snapsort_classes]
# CIFAR-10's own ids for these classes are scattered (e.g. "cat"=3, "ship"=8).
# CrossEntropyLoss needs compact 0..3 labels, so we remap: given, not today's focus.
snapsort_label_map = {original_id: compact_id for compact_id, original_id in enumerate(snapsort_class_ids)}

# Simulate scarcity: SnapSort claims ~60 labeled photos per class exist.
train_indices = make_balanced_indices(raw_train_full.targets, snapsort_class_ids, n_per_class=60, seed=SEED)
val_indices, test_indices = make_val_test_split(raw_test_full.targets, snapsort_class_ids, n_val=25, n_test=25, seed=SEED)

print("Train images:", len(train_indices))
print("Val images:", len(val_indices))
print("Test images:", len(test_indices))

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Deterministic preprocessing — تکمیل baseline transform</h2>
<p align="right" dir="rtl"><strong>TODO:</strong> <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_transform</code> زیر را کامل کنید تا هر image را به <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">64x64</code> resize کند، آن را به tensor تبدیل کند و دقیقاً مانند Session 1 و Session 2 normalize کند (<code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">mean=[0.5, 0.5, 0.5]</code> و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">std=[0.5, 0.5, 0.5]</code>؛ CIFAR-10 از نوع RGB است).</p>
<p align="right" dir="rtl">در این pipeline نباید هیچ randomness وجود داشته باشد، چون بعداً همین pipeline برای validation و test هم استفاده می‌شود.</p>

</div>


In [ ]:
baseline_transform = transforms.Compose([
    # TODO: transforms.Resize((?, ?))
    # TODO: transforms.ToTensor()
    # TODO: transforms.Normalize(mean=[...], std=[...])
])

train_baseline_dataset = TransformSubset(raw_train_full, train_indices, baseline_transform, label_map=snapsort_label_map)
val_dataset = TransformSubset(raw_test_full, val_indices, baseline_transform, label_map=snapsort_label_map)
test_dataset = TransformSubset(raw_test_full, test_indices, baseline_transform, label_map=snapsort_label_map)

class_names = snapsort_classes
print("Classes:", class_names)

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Visualize کردن یک batch — TODO</h2>
<p align="right" dir="rtl">loop زیر را کامل کنید؛ همان الگویی که در Session 1 برای batch visualization استفاده کردید. هر image را denormalize کنید، با <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">permute</code> به ترتیب <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">(H, W, C)</code> ببرید، مقدارها را به بازه <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">[0, 1]</code> clamp کنید و image را نمایش دهید؛ class name هم title آن باشد.</p>

</div>


In [ ]:
def denormalize(image_tensor):
    return image_tensor * 0.5 + 0.5


train_loader = DataLoader(train_baseline_dataset, batch_size=16, shuffle=True, num_workers=0)
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for image, label, axis in zip(images[:8], labels[:8], axes.flat):
    # TODO: denormalize `image`, permute to (H, W, C), clamp to [0, 1], imshow it
    # TODO: set the title to class_names[label.item()]
    axis.axis("off")

plt.tight_layout()
plt.show()

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>Q1.1:</strong> به batchای که همین الان plot کردید نگاه کنید. عکس‌های SnapSort از موبایل کاربران مختلف و در شرایط واقعی گرفته می‌شوند. <strong>دو</strong> منبع visual variation را که همین حالا در این sample کوچک 8تایی می‌بینید نام ببرید (مثلاً viewpoint، background، lighting یا scale) و یک مورد هم بگویید که فقط با دیدن 8 image نمی‌توان درباره‌اش قضاوت کرد.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>
<p align="right" dir="rtl"><strong>Q1.2:</strong> داشتن 60 image برای هر class در یک proof-of-concept برای یک startup در مرحله اولیه کاملاً عادی است. در یک یا دو جمله به product manager شرکت SnapSort توضیح دهید چه ریسکی دارد که model آموزش‌دیده با همین داده مستقیماً وارد production شود.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Train کردن baseline (فعلاً بدون augmentation)</h2>
<p align="right" dir="rtl">architecture زیر همان <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">SmallCNN</code> در Session 1/2 است که با argument به نام <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">in_channels</code> عمومی‌تر شده تا امروز بتوانیم آن را هم برای CIFAR-10 با 3 channel و هم برای MNIST با 1 channel استفاده کنیم.</p>

</div>


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.set_grad_enabled(is_training):
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = loss_fn(logits, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += batch_size

    return {"loss": total_loss / total_examples, "accuracy": total_correct / total_examples}


def make_loader(dataset, shuffle, batch_size=16, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0, generator=generator)


def train_experiment(train_dataset, val_dataset, num_classes, in_channels=3, epochs=15, seed=SEED):
    '''Reusable training loop — you will call this several times today for
    different client tickets. Keep every argument fixed except the dataset
    whenever you want a fair A/B comparison.'''
    set_seed(seed)

    train_loader = make_loader(train_dataset, shuffle=True, seed=seed)
    val_loader = make_loader(val_dataset, shuffle=False, seed=seed)

    model = SmallCNN(num_classes=num_classes, in_channels=in_channels).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    start_time = time.perf_counter()

    for epoch in range(epochs):
        train_metrics = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_accuracy"].append(val_metrics["accuracy"])

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | train acc: {train_metrics['accuracy']:.1%} | "
            f"val acc: {val_metrics['accuracy']:.1%}"
        )

    elapsed_seconds = time.perf_counter() - start_time
    return model, history, elapsed_seconds


def summarize_experiment(name, history, elapsed_seconds):
    best_epoch_index = int(np.argmin(history["val_loss"]))
    return {
        "experiment": name,
        "best_epoch": best_epoch_index + 1,
        "train_accuracy": history["train_accuracy"][best_epoch_index],
        "val_accuracy": history["val_accuracy"][best_epoch_index],
        "val_loss": history["val_loss"][best_epoch_index],
        "generalization_gap": history["train_accuracy"][best_epoch_index] - history["val_accuracy"][best_epoch_index],
        "seconds": elapsed_seconds,
    }

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>Q1.3 (قبل از اجرا پیش‌بینی کنید):</strong> با فقط 60 training image برای هر class و 4 class، یعنی در مجموع 240 image، انتظار دارید این baseline دچار overfitting شود؟ اگر بله، تقریباً از کدام epoch؟ پیش‌بینی خود را <strong>قبل از اجرای cell بعدی</strong> بنویسید؛ بلافاصله بعد از اجرا نتیجه را با حدستان مقایسه می‌کنید.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>
<p align="right" dir="rtl"><strong>TODO:</strong> <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_experiment</code> را با baseline datasetها، 4 class، مقدار <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">in_channels=3</code> و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">epochs=15</code> فراخوانی کنید.</p>

</div>


In [ ]:
# TODO: baseline_model, baseline_history, baseline_time = train_experiment(...)

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Plot و Diagnose</h2>
<p align="right" dir="rtl"><strong>TODO:</strong> برای baseline run، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_accuracy</code> و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">val_accuracy</code> را plot کنید (می‌توانید از الگوی plotting در Session 1 یا Session 2 استفاده کنید). سپس <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_history</code> را به function خودتان یعنی <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">diagnose_fit</code> از Part 0 بدهید و verdict را print کنید.</p>

</div>


In [ ]:
# TODO: plot baseline_history (accuracy, and optionally loss)

# TODO: print("diagnose_fit says:", diagnose_fit(baseline_history))

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>Q1.4:</strong> آیا نتیجه واقعی با پیش‌بینی Q1.3 شما مطابقت داشت؟ اگر نه، یک ویژگی مشخص از <strong>dataset واقعی</strong> را نام ببرید که intuition شما در Part 0، که بر اساس scenarioهای مصنوعی بود، آن را در نظر نگرفته بود.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h1 align="right" dir="rtl">Part 2 — طراحی Augmentation Policy برای SnapSort (≈35 دقیقه)</h1>
<p align="right" dir="rtl"><strong>اطلاعات بیشتر از ticket:</strong> کاربران SnapSort موبایل را با زاویه‌های مختلف نگه می‌دارند، از فاصله‌های متفاوت عکس می‌گیرند، هم در محیط داخل و هم بیرون عکاسی می‌کنند و معمولاً crop دقیقی ندارند. کار شما این نیست که یک «standard augmentation recipe» را اجرا کنید؛ باید برای هر transform جداگانه تصمیم بگیرید که آیا واقعاً باید در pipeline <strong>این محصول</strong> باشد یا نه.</p>
<h2 align="right" dir="rtl">Q2.1 — Decision table</h2>
<p align="right" dir="rtl">برای هر transform پیشنهادی مشخص کنید:</p>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" dir="rtl">آیا <strong>label را حفظ می‌کند</strong>؟</li>
<li align="right" dir="rtl">آیا برای یک عکس موبایلی از این object <strong>realistic</strong> است؟</li>
<li align="right" dir="rtl">آیا در <strong>deployment relevant</strong> است؛ یعنی کاربران واقعی واقعاً ممکن است چنین imageهایی تولید کنند؟</li>
</ul>
<p align="right" dir="rtl">در نهایت برای هر row یک <strong>Decision</strong> بگیرید: <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">Use</code>، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">Use cautiously (tune it down)</code> یا <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">Skip</code>، و در یک خط دلیل بنویسید.</p>
<table align="right" dir="rtl" style="direction: rtl; text-align: right; margin-left: auto; margin-right: 0;">
<thead align="right" dir="rtl">
<tr align="right" dir="rtl">
<th align="right" dir="rtl">Proposed transform</th>
<th align="right" dir="rtl" style="text-align:center">label را حفظ می‌کند؟</th>
<th align="right" dir="rtl" style="text-align:center">برای عکس موبایل realistic است؟</th>
<th align="right" dir="rtl" style="text-align:center">در deployment relevant است؟</th>
<th align="right" dir="rtl">Decision + دلیل</th>
</tr>
</thead>
<tbody align="right" dir="rtl">
<tr align="right" dir="rtl">
<td align="right" dir="rtl">Horizontal flip</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">Vertical flip</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">Small rotation (±15°)</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">Large rotation (±90°)</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">RandomResizedCrop (mild)</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">RandomResizedCrop (aggressive, up to 90% cropped out)</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">ColorJitter (brightness/contrast, mild)</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">Random grayscale</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
<tr align="right" dir="rtl">
<td align="right" dir="rtl">Random perspective warp</td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl" style="text-align:center"></td>
<td align="right" dir="rtl"></td>
</tr>
</tbody>
</table>
<p align="right" dir="rtl"><strong>Q2.2:</strong> از بین rowهای بالا، موردی را انتخاب کنید که درباره تصمیم آن کمترین اطمینان را داشتید. در دو جمله از Decision خود دفاع کنید؛ فرض کنید tech lead در stand-up از شما می‌پرسد «چرا؟».</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Implement کردن policy</h2>
<p align="right" dir="rtl"><strong>TODO:</strong> بر اساس table بالا، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">snapsort_augment</code> را فقط با transformهایی بسازید که تصمیم گرفتید <strong>نگه دارید</strong>. ترتیب تا حدی اهمیت دارد؛ یک convention رایج این است که crop/flip/rotate قبل از تغییرات color انجام شوند، هرچند قانون مطلقی نیست. فکر کنید چرا.</p>

</div>


In [ ]:
snapsort_augment = transforms.Compose([
    # TODO: add the transforms you decided to KEEP from the table in Q2.1,
    # in the order you think makes sense. Something like:
    # transforms.RandomResizedCrop(size=(64, 64), scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    # transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomRotation(degrees=...),
    # transforms.ColorJitter(...),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_augmented_dataset = TransformSubset(raw_train_full, train_indices, snapsort_augment, label_map=snapsort_label_map)

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">قبل از اعتماد، Visualize کنید</h2>
<p align="right" dir="rtl"><strong>TODO:</strong> از <strong>همان</strong> image ذخیره‌شده با index برابر 0، هشت augmented view متفاوت نمایش دهید. مانند Session 2، داخل یک loop هشت بار <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_augmented_dataset[0]</code> را بگیرید؛ هر بار randomness دوباره sample می‌شود. سپس imageها را denormalize و plot کنید.</p>

</div>


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
sample_index = 0

# TODO: for each of the 8 subplots:
#   image, label = train_augmented_dataset[sample_index]
#   denormalize -> permute -> clamp -> imshow
#   set title to class_names[label]

plt.suptitle("Eight random views of one stored image")
plt.tight_layout()
plt.show()

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Self-review checklist</h2>
<p align="right" dir="rtl">حداقل 3 class را بررسی کنید و از خودتان بپرسید :</p>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" dir="rtl">آیا object هنوز برای <strong>خود شما</strong> قابل تشخیص است؟</li>
<li align="right" dir="rtl">آیا این image می‌تواند واقعاً عکسی باشد که یک کاربر SnapSort گرفته است؟</li>
<li align="right" dir="rtl">آیا object در بعضی نمونه‌ها کاملاً از frame crop شده است؟</li>
<li align="right" dir="rtl">آیا یک class خاص به‌وضوح بیشتر از بقیه توسط policy آسیب می‌بیند؟</li>
</ul>
<p align="right" dir="rtl"><strong>Q2.3:</strong> آیا در augmented output خودتان چیزی دیدید که غافلگیرتان کند؛ مثلاً نتیجه‌ای که فقط با خواندن table در Q2.1 انتظار داشتید بهتر یا بدتر باشد؟</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h2 align="right" dir="rtl">Code review: pull request سینا</h2>
<p align="right" dir="rtl">هم‌تیمی شما، سینا، روی ticket مربوط به SnapSort کار کرده و pipeline زیر را برای review فرستاده است. پیام سینا:</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl">«augmentation را خیلی قوی‌تر کردم تا model تنوع بیشتری ببیند. همان transform را هم همه‌جا reuse کردم که مجبور نباشم دوبار بنویسم. آماده merge است؟»</p>
</blockquote>
<pre align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;"><code align="left" class="language-python" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">sina_transform = transforms.Compose([
    transforms.RandomVerticalFlip(p=1.0),
    transforms.RandomRotation(degrees=90),
    transforms.RandomResizedCrop(size=(64, 64), scale=(0.05, 1.0), ratio=(0.3, 3.0)),
    transforms.ColorJitter(brightness=0.9, contrast=0.9, saturation=0.9, hue=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = TransformSubset(raw_train_full, train_indices, sina_transform, label_map=snapsort_label_map)
val_dataset = TransformSubset(raw_test_full, val_indices, sina_transform, label_map=snapsort_label_map)
test_dataset = TransformSubset(raw_test_full, test_indices, sina_transform, label_map=snapsort_label_map)
</code></pre>
<p align="right" dir="rtl"><strong>Q2.4:</strong> حداقل <strong>سه bug یا تصمیم بدِ جداگانه</strong> در PR سینا پیدا کنید. فقط یک‌بار نگویید «عددها خیلی زیادند»؛ دقیقاً مشخص کنید <strong>کدام line</strong> مشکل دارد و <strong>چرا</strong>. در جاهایی که مرتبط است، به معیارهای label-preservation، realism و deployment-relevance در Q2.1 ارجاع دهید.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما (حداقل 3 مورد):</strong>
1.
2.
3.</p>
</blockquote>

</div>


In [ ]:
# TODO: write `corrected_transform`, a fixed version of Sina's pipeline.
# Reuse whatever you already trust from your own snapsort_augment where sensible,
# and make sure val/test are handled correctly.

corrected_transform = transforms.Compose([
    # TODO
])

# TODO: rebuild train_dataset / val_dataset / test_dataset correctly using
# corrected_transform for training and `baseline_transform` for val/test.
# Don't forget label_map=snapsort_label_map on every one of them.

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h1 align="right" dir="rtl">Part 3 — Ticket #2: SecureDigits، Augmentationای که نتیجه معکوس داد (≈40 دقیقه)</h1>
<p align="right" dir="rtl"><strong>متن ticket:</strong> SecureDigits اعداد دست‌نویس 0 تا 9 را از فرم‌های کاغذی scan‌شده می‌خواند. در یادداشت‌های contractor قبلی آمده است:</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl">«horizontal flip و full rotation را به training pipeline اضافه کردم؛ standard best practice برای robustتر کردن CNN نسبت به orientation.»</p>
</blockquote>
<p align="right" dir="rtl">بعد از deployment، accuracy در شرایط واقعی روی فرم‌های scan‌شده نسبت به نسخه بدون augmentation کاهش پیدا کرده است. tech lead از شما root-cause diagnosis می‌خواهد، نه حدس.</p>
<p align="right" dir="rtl">برای بررسی مسئله از subset کوچکی از <strong>MNIST</strong> استفاده می‌کنیم؛ datasetی کاملاً متفاوت با عکس‌های SnapSort. این تفاوت عمدی است: مهارتی که باید منتقل شود، یعنی بررسی label preservation قبل از زدن دکمه train، مهم است؛ نه حفظ‌کردن یک transform list خاص.</p>

</div>


In [ ]:
raw_mnist_train = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=None)
raw_mnist_test = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=None)

# Digits chosen on purpose: 6 and 9 are a classic flip/rotation trap; 2 and 7 give
# us a couple of "control" digits to see whether the damage is universal or specific.
digit_classes = [2, 3, 6, 7, 9]
digit_class_names = [str(d) for d in digit_classes]
# Same remapping trick as SnapSort: digit "6" and "9" aren't index 0/1, so we
# map {2:0, 3:1, 6:2, 7:3, 9:4} for CrossEntropyLoss. Given, not today's focus.
digit_label_map = {original_id: compact_id for compact_id, original_id in enumerate(digit_classes)}

digit_train_indices = make_balanced_indices(raw_mnist_train.targets.tolist(), digit_classes, n_per_class=80, seed=SEED)
digit_val_indices, digit_test_indices = make_val_test_split(
    raw_mnist_test.targets.tolist(), digit_classes, n_val=30, n_test=30, seed=SEED
)

print("Digit train images:", len(digit_train_indices))
print("Digit val images:", len(digit_val_indices))

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">اول raw data را ببینید</h2>
<p align="right" dir="rtl"><strong>TODO:</strong> deterministic MNIST transform را کامل کنید:
<code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">Resize((64, 64))</code>، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">ToTensor()</code> و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">Normalize(mean=[0.5], std=[0.5])</code>.</p>
<p align="right" dir="rtl">توجه کنید MNIST از نوع single-channel است، بنابراین listهای mean/std طول 1 دارند نه 3. سپس چند raw example از digit <strong>6</strong> و digit <strong>9</strong> را کنار هم plot کنید تا قبل از هر transform یک تصویر ذهنی واضح از داده داشته باشید.</p>

</div>


In [ ]:
digits_baseline_transform = transforms.Compose([
    # TODO: Resize((64, 64))
    # TODO: ToTensor()
    # TODO: Normalize(mean=[?], std=[?])  <- one value each, MNIST has 1 channel
])

digits_train_baseline = TransformSubset(raw_mnist_train, digit_train_indices, digits_baseline_transform, label_map=digit_label_map)
digits_val_dataset = TransformSubset(raw_mnist_test, digit_val_indices, digits_baseline_transform, label_map=digit_label_map)
digits_test_dataset = TransformSubset(raw_mnist_test, digit_test_indices, digits_baseline_transform, label_map=digit_label_map)


def denormalize_gray(image_tensor):
    return image_tensor * 0.5 + 0.5


# Find one raw 6 and one raw 9 to compare
six_index = next(i for i in digit_train_indices if raw_mnist_train.targets[i].item() == 6)
nine_index = next(i for i in digit_train_indices if raw_mnist_train.targets[i].item() == 9)

fig, axes = plt.subplots(1, 2, figsize=(5, 3))
for axis, idx, label in zip(axes, [six_index, nine_index], ["6", "9"]):
    image, _ = raw_mnist_train[idx]
    axis.imshow(image, cmap="gray")
    axis.set_title(f"Original digit {label}")
    axis.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">transform مربوط به contractor را بازسازی کنید</h2>
<p align="right" dir="rtl"><strong>TODO:</strong> <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">harmful_digit_transform</code> را دقیقاً مطابق یادداشت contractor بسازید: ابتدا horizontal flip با <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">p=1.0</code> تا در این demo اثر آن را به‌صورت deterministic ببینیم (هرچند در یک pipeline واقعی معمولاً <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">p=0.5</code> استفاده می‌شود)، و بعد full <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">RandomRotation(degrees=180)</code>.</p>
<p align="right" dir="rtl">سپس این transform را روی <strong>همان digit-6 image</strong> که قبلاً دیدید اعمال کنید و نتیجه را کنار original نمایش دهید.</p>

</div>


In [ ]:
harmful_digit_transform = transforms.Compose([
    # TODO: transforms.RandomHorizontalFlip(p=1.0)
    # TODO: transforms.RandomRotation(degrees=180)
])

original_six, _ = raw_mnist_train[six_index]
# TODO: augmented_six = harmful_digit_transform(original_six)

fig, axes = plt.subplots(1, 2, figsize=(5, 3))
axes[0].imshow(original_six, cmap="gray")
axes[0].set_title("Original (label: 6)")
axes[0].axis("off")
# TODO: axes[1].imshow(augmented_six, cmap="gray")
axes[1].set_title("After contractor's transform (label still says: 6)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>Q3.1:</strong> «augmented 6» حالا از نظر ظاهری شبیه کدام digit شده است؟ label ذخیره‌شده کنار این image هنوز <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">6</code> است. در یک جمله توضیح دهید چرا این دقیقاً همان failure modeای است که معیار <strong>label-preservation</strong> در table مربوط به Part 2 باید آن را پیدا کند.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>
<p align="right" dir="rtl"><strong>Q3.2:</strong> digitهای 6 و 9 قربانی‌های واضح این transform هستند. از بین سه digit دیگر در subset ما یعنی <strong>2، 3 و 7</strong>، فکر می‌کنید کدام مورد حتی بدون flip هم با rotation برابر 180° آسیب می‌بیند؟ کدام‌یک تقریباً safe است؟ پاسخ را بر اساس شکل ظاهری هر digit وقتی وارونه می‌شود توجیه کنید.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">میزان آسیب را Quantify کنید — فقط به چشم اعتماد نکنید</h2>
<p align="right" dir="rtl">دیدن یک «6» خراب‌شده قانع‌کننده است، اما tech lead از شما عدد می‌خواهد.</p>
<p align="right" dir="rtl"><strong>TODO:</strong> دو experiment را روی <strong>دقیقاً همان digit data</strong> train کنید و فقط training transform را تغییر دهید:</p>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" dir="rtl"><strong>Experiment "safe"</strong> — فقط <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">digits_baseline_transform</code> و بدون هیچ randomness.</li>
<li align="right" dir="rtl"><strong>Experiment "harmful"</strong> — <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">harmful_digit_transform</code>. همان flip+rotation pipeline بالا را reuse کنید، اما برای flip مقدار <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">p=1.0</code> را به مقدار realisticتر یعنی <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">p=0.5</code> برگردانید تا مثل یک training pipeline واقعی رفتار کند، نه demo deterministic بالا.</li>
</ul>
<p align="right" dir="rtl">بقیه چیزها باید کاملاً یکسان باشند: همان <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_experiment</code>، همان <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">epochs</code> و همان <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">seed</code>.</p>

</div>


In [ ]:
harmful_digit_train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

digits_train_harmful = TransformSubset(raw_mnist_train, digit_train_indices, harmful_digit_train_transform, label_map=digit_label_map)

# TODO: safe_model, safe_history, safe_time = train_experiment(
#     digits_train_baseline, digits_val_dataset, num_classes=len(digit_classes),
#     in_channels=1, epochs=12,
# )

# TODO: harmful_model, harmful_history, harmful_time = train_experiment(
#     digits_train_harmful, digits_val_dataset, num_classes=len(digit_classes),
#     in_channels=1, epochs=12,
# )

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>TODO:</strong> هر دو run را با <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">summarize_experiment</code> خلاصه کنید و یک bar chart ساده بسازید که best validation accuracy را برای safe و harmful با هم مقایسه کند.</p>

</div>


In [ ]:
# TODO: safe_summary = summarize_experiment("Safe (no augmentation)", safe_history, safe_time)
# TODO: harmful_summary = summarize_experiment("Harmful (flip + 180 rotation)", harmful_history, harmful_time)
# TODO: print(safe_summary); print(harmful_summary)

# TODO: bar chart — x labels ["Safe", "Harmful"], heights = the two val_accuracy values

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>Q3.3 (incident report):</strong> یک paragraph دو تا سه‌جمله‌ای بنویسید که واقعاً می‌توانستید در incident report مربوط به SecureDigits قرار دهید. به عددهای واقعی خودتان از <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">safe_summary</code> و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">harmful_summary</code> ارجاع دهید، classهایی را که فکر می‌کنید بیشترین نقش را در آسیب دارند نام ببرید و مشخص کنید به client پیشنهاد می‌دهید قدم بعدی چه باشد.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">حالا policyای طراحی کنید که واقعاً به SecureDigits کمک کند</h2>
<p align="right" dir="rtl">فرم‌های واقعی scan‌شده variationهایی دارند که ارزش train کردن دارند: کمی کج‌شدن کاغذ، تغییرات جزئی در smudge/contrast و تفاوت کوچک در اندازه handwriting. اما full rotation یا mirroring جزو این variationهای واقعی نیستند.</p>
<p align="right" dir="rtl"><strong>TODO:</strong> <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">safe_digit_augment</code> را فقط با transformهای mild و label-preserving بسازید؛ مثلاً یک <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">RandomRotation</code> کوچک (با توجه به نتیجه Q3.2 یک محدوده degree منطقی انتخاب کنید) و/یا <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">RandomAffine</code> با translation کم. هیچ flipای استفاده نکنید.</p>
<p align="right" dir="rtl">سپس 8 view از همان digit-6 image را visualize کنید تا نشان دهید در همه حالت‌ها هنوز شبیه 6 باقی می‌ماند.</p>

</div>


In [ ]:
safe_digit_augment = transforms.Compose([
    transforms.Resize((64, 64)),
    # TODO: add your label-preserving transform(s) here, e.g.
    # transforms.RandomRotation(degrees=...),
    # transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

digits_train_safe_aug = TransformSubset(raw_mnist_train, digit_train_indices, safe_digit_augment, label_map=digit_label_map)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
# TODO: plot 8 augmented views of digits_train_safe_aug at the index of a digit-6
#       example (find its position within digit_train_indices), same pattern as before
plt.suptitle("Eight views of a digit 6 under the SAFE policy — still a 6 every time")
plt.tight_layout()
plt.show()

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h1 align="right" dir="rtl">Part 4 — Flickering Metric: چرا Validation Data باید Deterministic بماند (≈25 دقیقه)</h1>
<p align="right" dir="rtl">همین حالا پیامی در Slack تیم از یکی از هم‌دوره‌ای‌های bootcamp شما که او هم junior engineer شده رسیده است:</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>Deniz, 10:41 AM:</strong> «اگر random augmentation باعث robustتر شدن training می‌شود، نباید validation set را هم augment کنیم؟ این‌طوری robustness را هم test می‌کنیم، نه فقط imageهای clean و حفظ‌شده را. می‌خواهم <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">RandomHorizontalFlip</code> را به val transform اضافه کنم؛ نظرت چیست؟»</p>
</blockquote>
<p align="right" dir="rtl">از روی حافظه جواب ندهید. اول با استفاده از <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_model</code> train‌شده خودتان در Part 1 شواهد بسازید.</p>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>TODO:</strong> functionای به نام <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">evaluate_with_seed</code> بنویسید که یک validation loader را با <strong>transform و seed مشخص</strong> بسازد و با استفاده از helper موجود یعنی <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">run_epoch</code> یک evaluation pass از نوع deterministic-vs-random اجرا کند؛ بدون optimizer.</p>
<p align="right" dir="rtl">سپس:</p>
<ol align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" dir="rtl"><code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_model</code> را فقط <strong>یک‌بار</strong> روی <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">val_dataset</code>، یعنی validation set واقعی و deterministic، evaluate کنید و این مقدار را reference accuracy بنامید.</li>
<li align="right" dir="rtl">به‌جای <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_transform</code>، با استفاده از <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">snapsort_augment</code> یا هر transform دارای randomness واقعی یک «flip-augmented validation set» بسازید. سپس <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_model</code> را <strong>5 بار جداگانه با 5 seed متفاوت</strong> روی آن evaluate کنید. خود model در این 5 run هیچ تغییری نمی‌کند.</li>
</ol>

</div>


In [ ]:
def evaluate_with_seed(model, base_dataset_indices, base_dataset_source, transform, seed, label_map=snapsort_label_map):
    generator_seed = seed
    dataset = TransformSubset(base_dataset_source, base_dataset_indices, transform, label_map=label_map)
    loader = make_loader(dataset, shuffle=False, seed=generator_seed)
    loss_fn = nn.CrossEntropyLoss()
    # TODO: call run_epoch(model, loader, loss_fn) with no optimizer and return its "accuracy"
    raise NotImplementedError


# Reference: the real, deterministic validation pipeline — one honest number.
# TODO: reference_accuracy = evaluate_with_seed(baseline_model, val_indices, raw_test_full, baseline_transform, seed=SEED)

# Now the "robustness test" Deniz proposed: same model, same images, randomized transform.
flickering_accuracies = []
for trial_seed in [1, 2, 3, 4, 5]:
    # TODO: acc = evaluate_with_seed(baseline_model, val_indices, raw_test_full, snapsort_augment, seed=trial_seed)
    # TODO: flickering_accuracies.append(acc)
    pass

print("Reference (deterministic) accuracy:", reference_accuracy)
print("Five 'robustness test' accuracies:", flickering_accuracies)

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>TODO:</strong> پنج randomized-validation accuracy را به‌صورت point یا یک bar chart کوچک plot کنید و یک horizontal line هم برای deterministic reference accuracy قرار دهید.</p>

</div>


In [ ]:
# TODO: plt.scatter or plt.bar for flickering_accuracies
# TODO: plt.axhline(reference_accuracy, linestyle="--", label="deterministic reference")
# TODO: labels, legend, show

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>Q4.1:</strong> weightهای model در پنج «robustness test» شما هیچ تغییری نکردند؛ همان <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_model</code> و همان 100 validation image استفاده شد. پس چرا مقدار accuracy همچنان تغییر می‌کرد؟ با زبان خودتان توضیح دهید اگر از این randomized pipeline برای انتخاب best checkpoint استفاده می‌کردید، این نوسان چه مشکلی برای مقایسه «epoch 7» با «epoch 12» ایجاد می‌کرد.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>
<p align="right" dir="rtl"><strong>Q4.2 — پاسخ به Deniz:</strong> همان Slack messageای را بنویسید که واقعاً برای Deniz می‌فرستادید؛ 3 تا 4 جمله، دوستانه اما از نظر فنی دقیق. به‌جای گفتن فقط یک قانون کلی، از عددهای experiment خودتان به‌عنوان evidence استفاده کنید.</p>
<p align="right" dir="rtl">اگر فکر می‌کنید evaluate کردن model روی augmented imageها <strong>در یک موقعیت دیگر</strong> واقعاً مفید است، به آن اشاره کنید. Hint: این یک technique جداگانه برای test کردن robustness است و با استفاده از validation برای انتخاب checkpoint یا گزارش headline metric فرق دارد. دقیق توضیح دهید چرا این دو موقعیت یکی نیستند.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>پاسخ شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Bonus: ساخت pre-deployment auditor</h2>
<p align="right" dir="rtl">تیم‌های واقعی ML معمولاً یک check خودکار به CI اضافه می‌کنند تا چنین اشتباهی بی‌سروصدا وارد production نشود.</p>
<p align="right" dir="rtl"><strong>TODO:</strong> <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">audit_pipeline(transform)</code> را بنویسید؛ این function باید یک <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">transforms.Compose</code> را inspect کند و فقط زمانی <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">True</code> برگرداند که <strong>هیچ‌کدام</strong> از random transform classهای شناخته‌شده داخل آن نباشند.</p>
<p align="right" dir="rtl">آن را روی <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_transform</code> اجرا کنید که باید pass شود، و روی <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">snapsort_augment</code> که باید fail شود؛ این fail درست است چون <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">snapsort_augment</code> یک <strong>training pipeline</strong> است.</p>

</div>


In [ ]:
RANDOM_TRANSFORM_NAMES = {
    "RandomHorizontalFlip", "RandomVerticalFlip", "RandomRotation",
    "RandomResizedCrop", "ColorJitter", "RandomAffine", "RandomPerspective",
    "RandomGrayscale", "RandomCrop", "RandomErasing",
}


def audit_pipeline(transform):
    '''Return True if `transform` (a transforms.Compose) contains ZERO
    random operations — i.e. it is safe to use for validation/test/deployment.'''
    # TODO: inspect transform.transforms (a list of transform objects),
    #       check each one's class name (type(t).__name__) against RANDOM_TRANSFORM_NAMES
    raise NotImplementedError


print("baseline_transform safe for eval?", audit_pipeline(baseline_transform))
print("snapsort_augment safe for eval?  ", audit_pipeline(snapsort_augment))

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h1 align="right" dir="rtl">Part 5 — بستن Ticket #1: اول اثبات کنید، بعد به Client گزارش دهید (≈35 دقیقه)</h1>
<p align="right" dir="rtl">حالا باید دقیقاً همان چیزی را تمام کنید که SnapSort خواسته بود: آیا augmentation policy شما، یعنی نسخه اصلاح‌شده Part 2، واقعاً کمک می‌کند یا فقط در visualization خوب به‌نظر می‌رسد؟</p>
<p align="right" dir="rtl">قبل از اجرا، یک تیم واقعی ML معمولاً experiment را «pre-register» می‌کند؛ یعنی از قبل مشخص می‌کند چه چیزهایی باید ثابت بمانند تا کسی بعداً بی‌سروصدا معیار مقایسه را تغییر ندهد. تأیید کنید هر checkbox زیر با setup واقعی شما مطابقت دارد:</p>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>همان train/val/test splitها (<code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_indices</code>, <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">val_indices</code>, <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">test_indices</code>)</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>همان model architecture (<code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">SmallCNN</code>)</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>همان optimizer و learning rate یعنی Adam با <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">lr=1e-3</code> که داخل <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_experiment</code> تنظیم شده</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>همان تعداد epoch برای هر دو run</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>همان batch size</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>همان random seed ارسال‌شده به <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">train_experiment</code></li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>validation pipeline کاملاً یکسان و deterministic برای هر دو run (<code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">baseline_transform</code>)</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/><strong>تنها تفاوت، training transform است.</strong></li>
</ul>

</div>


In [ ]:
# TODO: run the two experiments with everything fixed except the training transform.
# snapsort_baseline_dataset = TransformSubset(raw_train_full, train_indices, baseline_transform, label_map=snapsort_label_map)
# snapsort_corrected_dataset = TransformSubset(raw_train_full, train_indices, corrected_transform, label_map=snapsort_label_map)

# TODO: snap_baseline_model, snap_baseline_history, snap_baseline_time = train_experiment(
#     snapsort_baseline_dataset, val_dataset, num_classes=4, in_channels=3, epochs=15,
# )

# TODO: snap_aug_model, snap_aug_history, snap_aug_time = train_experiment(
#     snapsort_corrected_dataset, val_dataset, num_classes=4, in_channels=3, epochs=15,
# )

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>TODO:</strong> هر چهار accuracy curve را روی یک axes plot کنید: baseline train/val و augmented train/val. از همان style مربوط به comparison plot در Session 2 استفاده کنید.</p>

</div>


In [ ]:
# TODO: epochs_range = range(1, 16)
# TODO: plot baseline train/val accuracy AND augmented train/val accuracy together
# TODO: optionally, a second subplot with both validation-loss curves

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<p align="right" dir="rtl"><strong>TODO:</strong> برای هر دو history از <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">summarize_experiment</code> استفاده کنید و یک مقایسه side-by-side را print کنید.</p>

</div>


In [ ]:
# TODO: snap_baseline_summary = summarize_experiment("A - Baseline", snap_baseline_history, snap_baseline_time)
# TODO: snap_aug_summary = summarize_experiment("B - Augmented", snap_aug_history, snap_aug_time)
# TODO: print both summaries

<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<h2 align="right" dir="rtl">Client memo</h2>
<p align="right" dir="rtl"><strong>TODO (این بخش نوشتاری است، code ننویسید):</strong> memo زیر را با استفاده از عددهای واقعی خودتان کامل کنید. متن را حدود 150 تا 200 کلمه نگه دارید؛ یک PM واقعی احتمالاً متن طولانی‌تر را نمی‌خواند.</p>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>To:</strong> SnapSort product team<br/>
<strong>From:</strong> [نام شما], ML Engineering<br/>
<strong>Re:</strong> Augmentation experiment results</p>
<p align="right" dir="rtl">ما دو نسخه از proof-of-concept classifier را روی همان 240 training image مقایسه کردیم و تنها چیزی که تغییر دادیم، image transformationهای زمان training بود.</p>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" dir="rtl">Baseline best validation accuracy: <strong>[تکمیل کنید]</strong> (epoch <strong>[تکمیل کنید]</strong>)</li>
<li align="right" dir="rtl">Augmented best validation accuracy: <strong>[تکمیل کنید]</strong> (epoch <strong>[تکمیل کنید]</strong>)</li>
<li align="right" dir="rtl">Generalization gap (train − val accuracy) برای baseline در مقایسه با augmented: <strong>[تکمیل کنید]</strong></li>
</ul>
<p align="right" dir="rtl">[در یک یا دو جمله توضیح دهید augmentation کمک کرد، نتیجه را بدتر کرد یا تفاوت واضحی ایجاد نکرد. حتماً به عددهای بالا ارجاع دهید.]</p>
<p align="right" dir="rtl"><strong>Caveat:</strong> [حداقل یک limitation را ذکر کنید؛ مثلاً single random seed، validation set کوچک با فقط 25 image برای هر class، تعداد محدود epoch یا هر مورد دیگری که مشاهده کردید.]</p>
<p align="right" dir="rtl"><strong>Recommendation:</strong> [آیا model به همین شکل Ship شود؟ قبل از shipping داده واقعی بیشتری جمع شود؟ policy قوی‌تر/ضعیف‌تر امتحان شود؟ قبل از تصمیم‌گیری experiment با چند seed تکرار شود؟ یکی را انتخاب و در یک جمله توجیه کنید.]</p>
</blockquote>
<blockquote align="right" dir="rtl">
<p align="right" dir="rtl"><strong>Memo شما:</strong></p>
</blockquote>

</div>


<div dir="rtl" align="right" style="direction: rtl; text-align: right; line-height: 1.9;">
<hr/>
<h1 align="right" dir="rtl">Part 6 — Stretch Goals (اختیاری، اگر زودتر تمام کردید)</h1>
<p align="right" dir="rtl">می‌توانید هرکدام از موارد زیر را انتخاب کنید. انجام آن‌ها الزامی نیست، اما هر مورد مستقیماً به ایده‌ای از بخش <strong>If Augmentation Does Not Help</strong> در Session 2 مرتبط است.</p>
<ol align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" dir="rtl"><p align="right" dir="rtl"><strong>آیا نتیجه شما واقعاً معنی‌دار است یا فقط noise؟</strong> مقایسه SnapSort baseline-vs-augmented را با 2 random seed دیگر دوباره اجرا کنید تا در مجموع 3 seed داشته باشید. به‌جای گزارش یک عدد، <strong>mean و standard deviation</strong> مربوط به best validation accuracy هر نسخه را گزارش کنید. آیا نتیجه‌ای که در Part 5 گرفتید همچنان برقرار است؟</p>
</li>
<li align="right" dir="rtl"><p align="right" dir="rtl"><strong>Test-time augmentation (TTA) — آشنایی با یک technique واقعی.</strong> TTA با پیشنهادی که Deniz در Part 4 داد فرق دارد. به‌جای استفاده از randomized validation set برای <strong>انتخاب checkpoint</strong>، که همان اشتباه است، یک model <strong>already-selected و already-frozen</strong> را برمی‌دارید و فقط در prediction time روی imageهای جدید، predictionهای چند augmented view از <strong>همان test image</strong> را با هم average می‌کنید. سپس prediction نهایی را با single ground-truth label مقایسه می‌کنید. هیچ labelای وارد augmentation نمی‌شود و هیچ training decisionای به آن وابسته نیست. امتحان کنید: برای 10 test image، softmax probability را روی 5 augmented view از هر image average کنید و accuracy را با single-view baseline prediction مقایسه کنید.</p>
</li>
<li align="right" dir="rtl"><p align="right" dir="rtl"><strong>مطالعه یک alternative: MixUp.</strong> MixUp imageها را rotate یا crop نمی‌کند؛ دو training image و labelهای آن‌ها را با هم blend می‌کند. برای مثال، یک image شامل 70% cat و 30% dog می‌تواند labelای شامل 70% cat و 30% dog داشته باشد. یک paragraph از یک منبع معتبر درباره MixUp پیدا کنید و در 3 تا 4 جمله <strong>با بیان خودتان</strong> توضیح دهید چه تفاوتی با همه transformهایی دارد که امروز طراحی کردید و چرا سؤال «آیا این transform label را حفظ می‌کند؟» برای MixUp کمی معنای متفاوتی نسبت به rotation دارد.</p>
</li>
</ol>
<hr/>
<h1 align="right" dir="rtl">Submission Checklist</h1>
<ul align="right" dir="rtl" style="padding-right: 1.6em; padding-left: 0;">
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>Part 0: <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">diagnose_fit</code> پیاده‌سازی شده و Q0.1/Q0.2 پاسخ داده شده</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>Part 1: baseline transform تکمیل شده، baseline train شده و Q1.1–Q1.4 پاسخ داده شده</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>Part 2: decision table تکمیل شده، <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">snapsort_augment</code> پیاده‌سازی و visualize شده، حداقل 3 bug سینا پیدا شده و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">corrected_transform</code> نوشته شده</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>Part 3: harmful transform بازسازی و visualize شده، experiment مربوط به safe-vs-harmful اجرا و مقایسه شده، incident-report paragraph نوشته شده و safe policy طراحی و visualize شده</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>Part 4: flickering-metric experiment اجرا و plot شده، پاسخ Deniz نوشته شده و <code align="left" dir="ltr" style="direction: ltr; text-align: left; unicode-bidi: isolate;">audit_pipeline</code> پیاده‌سازی شده</li>
<li align="right" class="task-list-item" dir="rtl"><input class="task-list-item-checkbox" disabled="" type="checkbox"/>Part 5: final controlled experiment اجرا، plot و summarize شده و client memo نوشته شده</li>
</ul>
<p align="right" dir="rtl">خسته نباشید؛ شما یک augmentation workflow کامل را از ابتدا تا انتها روی دو مسئله با شکل واقعی انجام دادید.</p>

</div>
